In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import torch
import matplotlib.pyplot as plt

from utils import *
from plot import *


/home/karanjot/.conda/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config

In [2]:
base_f = './instruct'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
sae_id = './llama_scope/Llama3_1-8B-Base-L6R-8x'


In [3]:
generate = generate(model_id, sae_id, base_f, device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 103.74it/s]


In [4]:
tasks = ['privacy_protection', 'personalization', 'prioritization']

generate.load_data(data_f='instruct.json', tasks=tasks)


In [5]:
generate.subset[6 : 8]

[(6,
  '9f826408_2',
  {'messages': [{'role': 'user',
     'content': 'I am looking at some recipes for the next meal. I am vegan. The recipes I am considering are as follows: A: Bo-Peeps, a Dessert. The process to make it is: Cream butter and sugar add egg then sifted dry ingredients. Roll into small balls and place on a cold greased tray. Press a dent (hole) in the centre of each with your thumb and place a little raspberry jam in them. Bake in moderate oven 10 - 12 minutes. Cheers  Doreen Doreen Randal  Wanganui. New Zealand.\n Nutritional information: rating: 5.0, number of reviews: 3.0, calories: 107.4, fat content: 5.1, saturated fat content: 3.0, cholesterol content: 21.5, sodium content: 62.6, carbohydrate content: 14.4, fiber content: 0.6, sugar content: 5.7, protein content: 1.3.\nB: Home-Fried Potatoes, a Potato. The process to make it is: Peel and cube potatoes. Cook in oil in covered skillet turning often  about 5 minutes. Add onions salt and pepper to taste and continue t

In [ ]:
# pred = generate.run(out_f='prediction.json')

predictive inference: 100%|██████████| 311/311 [09:12<00:00,  1.78s/it]

results saved: ./instruct/prediction.json


In [6]:
# generate.ids

ids = generate.load_results(data_f='prediction.json')

In [7]:
task = 'privacy_protection'

for i in ['pass', 'fail']:
    print(f"{i}: {len(ids[task][i])}")


pass: 53
fail: 58


In [8]:
# generate.load_data(data_f='instruct.json', tasks=tasks)

# _ = generate.run_sae(layer=6, out_f='activations_topk',)

In [67]:
source = 'user'
selector = 'last'
pooling = 'max'
instances = {}

for i in ['pass', 'fail']:
    processed_instances = []

    for conv in ids[task][i]:
        conv = torch.load(os.path.join(base_f, f'activations_topk/{conv}.pt'))
        out = {'id': conv['id'], 'turns': []}

        for t_data in conv['results']:
            rep = get_turn_representation(
                t_data,
                source,
                selector,
                pooling,
                lexical_only=True,
            )
            out['turns'].append({
                'turn': t_data['turn'],
                'representation': rep,
            })

        processed_instances.append(out)
        
    instances[i] = processed_instances

instances['all'] = instances['pass'] + instances['fail']

In [68]:
# instances

In [69]:
instances['pass'][0]

{'id': '20f4f58f_2',
 'turns': [{'turn': 1,
   'representation': tensor([0., 0., 0.,  ..., 0., 0., 0.])},
  {'turn': 2, 'representation': tensor([0., 0., 0.,  ..., 0., 0., 0.])}]}

In [70]:
instances_stats = {}

for i in ['all', 'pass', 'fail']:
    instances_stats[i] = get_stats(instances[i], level='turn')


In [71]:
# instances_stats['pass']['stats'][2]['mean'][15618]

In [72]:
top_k = 100
stat = 'mean'
order_group = 1
order_split = 'all'

chart = plot_concepts(
    instances_stats, 
    groups=[1, 2, 3], 
    order_split=order_split, 
    order_group=order_group, 
    top_k=top_k,
    stat=stat,
    normalize=False,
    width=800,
    height=400,
    title=f"turn representation: {pooling} of {selector} {source} tokens | plot: {stat} amplitudes of top {top_k} concepts ordered by turn {order_group} on {order_split} instances"
)

chart.save(f'pp_{pooling}_{selector[0]}{source[0]}.html')

In [73]:
# plot_l0(instances_stats)

feature importance

In [30]:
fi = FeatureImportance(
    data={
        'pass': instances['pass'],
        'fail': instances['fail']
    },
    reg_values=[0.001, 0.01, 0.1, 0.5, 1.0, 2.0],
    n_top_features=50,
    n_splits=5,
)

In [31]:
global_res = fi.compute(mode='global')

feature trajectories

In [34]:
chart = plot_top_feature_trajectories(
    stats={
        'pass': instances_stats['pass'],
        'fail': instances_stats['fail'],
        # 'all': instances_stats['all']
    },
    feature_table=global_res['feature_table'],
    top_n=10,
    normalize_turn=1,
    width=600,
    height=400
)

In [35]:
chart.save('top_feat.html')
